In [1]:
import sys
from pathlib import Path

PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import config
from rx import io, text, places, remote
print(config.PROJECT_DIR)

/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads


In [2]:
con = remote.connect()
PENSION = remote.hf_url("pension")
print(PENSION)

con.sql(f"""
        select count(*) as pages,
        count(distinct NAID) as files
        from '{PENSION}'
""").df()

hf://datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files@90f90867333c320af57a2ed1f43d5187c6685893/data/train-*.parquet


,pages,files
0,2195663,78745


In [3]:
files_raw = con.sql(f"""
                    select NAID, any_value(title) as title, count(*) as n_pages
                    from '{PENSION}'
                    group by NAID
""").df()
io.write_stage(files_raw, "interim", "files_raw")
files_raw.shape

(78745, 3)

In [4]:
pages_index = con.sql(f"""
    select NAID, pageNumber,
           coalesce(trim(transcriptionText) <> '', false) as has_transcription,
           coalesce(trim(RevX_OCR_text)     <> '', false) as has_revx_ocr,
           coalesce(trim(fs_extractedText)  <> '', false) as has_fs_text
    from '{PENSION}'
""").df()
io.write_stage(pages_index, "interim", "pages_index")
pages_index.shape


(2195663, 5)

In [5]:
TXT = "lower(coalesce(nullif(trim(TranscriptionText), ''), nullif(trim(RevX_OCR_text), ''), fs_extractedText))"
pages_decl = con.sql(f"""
                     select NAID, pageNumber, transcriptionText, RevX_OCR_text, fs_extractedText
                     from '{PENSION}'
                     where {TXT} like '%personally appeared%'
                     qualify row_number() over (partition by NAID order by pageNumber) = 1
                     """).df()
io.write_stage(pages_decl, "interim", "pages_decl")
pages_decl.shape

(62112, 5)

In [6]:
state_mentions = con.sql(f"""
                         select NAID, state_raw, count(*) as n
                         from (
                             select NAID, 
                             unnest(regexp_extract_all({TXT}, 'state of ((?:north |south |new )?[a-z]+)', 1)) as state_raw
                             from '{PENSION}'
                             )
                             group by NAID, state_raw
                             """).df()
io.write_stage(state_mentions, "interim", "state_mentions")
state_mentions.shape

(204846, 3)

In [7]:
assert files_raw.NAID.is_unique and len(files_raw) == 78745
assert files_raw.n_pages.sum() == len(pages_index) == 2195663
assert pages_decl.NAID.is_unique
assert pages_decl.NAID.isin(files_raw.NAID).all()
print("A01 ok")

A01 ok
